In [0]:
from pyspark.sql import functions as F
base_rate = spark.table("scottish_housing_ws.silver.boe_base_rate")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
from pyspark.sql.window import Window
gold = (
    hpi
    .join(base_rate.select("year_month", "base_rate_pct"), on="year_month", how="inner")
    .withColumn(
        "rate_change_pct",
        F.round(
            F.col("base_rate_pct") - F.lag("base_rate_pct").over(
                Window.partitionBy("area_code").orderBy("report_date")
            ), 2
        )
    )
    .select(
        "report_date",
        "year_month",
        "region_name",
        "area_code",
        "average_price",
        "sales_volume",
        "base_rate_pct",
        "rate_change_pct"
    )
    .orderBy("area_code", "report_date")
)

In [0]:

print(f"Row count: {gold.count()}")
gold.filter(F.col("area_code") == "S92000003").orderBy("report_date").show(5)
gold.filter(F.col("area_code") == "S92000003").orderBy(F.col("report_date").desc()).show(5)


In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.base_rate_vs_transaction_volume")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_base_rate_vs_transaction_volume/")
)

print("Gold 03 complete.")